In [1]:
import sys

sys.path.append("/home/arota/Match/")
import os
import torch
import yaml
from dotmap import DotMap
from dataset import CHOLEC80, SCARED
from pipelines.odometry.odometry import OdometryPipeline
from pipelines.depth.depth import DepthPipeline
from pipelines.matching.matching import MatchingPipeline
from pipelines.geometry.geometry import GeometryPipeline
from pipelines.depth.tsdf import TSDFVolume
import matplotlib.pyplot as plt
from utilities.visualization import rgb, panelize
import numpy as np
from dotenv import load_dotenv

load_dotenv()
%load_ext autoreload
%autoreload 2

In [2]:
CONFIG_PATH = "config_train.yaml"
with open(CONFIG_PATH, "r") as f:
    config_yaml = yaml.safe_load(f)
    config_parameters = config_yaml["parameters"]
    config_training_dict = {
        k: v.get("value") for k, v in config_parameters.items() if v is not None
    }
    config = DotMap(config_training_dict)
# config.TRIPLETS_TO_MINE = 64

In [3]:
dataset = SCARED(
    path="/home/shared/nearmrs/arota/SCARED",
    # vids=["v33"],
    # exclude=["val_"],
    frameskip=[32],
    fps=32,
    random_pose=False,
    height=config.IMAGE_HEIGHT,
    width=config.IMAGE_WIDTH,
    with_depth=False,
    unit_translation=False,
)
len(dataset)


25481

In [4]:
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1)
iloader = iter(dataloader)
sample = next(iloader)
# Print sample keys and shapes
for k, v in sample.items():
    if hasattr(v, "shape"):
        print(f"{k}: Tensor of shape {list(v.shape)}")
    else:
        print(f"{k}: {type(v)}")


idx: Tensor of shape [1]
framestack: Tensor of shape [1, 2, 3, 384, 384]
Ts2t: Tensor of shape [1, 6]
paths: <class 'list'>
frameskip: Tensor of shape [1]
fundamental: Tensor of shape [1, 1, 3, 3]


In [5]:
# Initialize device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize Geometry Pipeline (for depth + normals + intrinsics)
geometryPipeline = GeometryPipeline(
    geometry_model_name="Ruicheng/moge-2-vits-normal",
    device=device,
    height=config.IMAGE_HEIGHT,
    width=config.IMAGE_WIDTH,
    return_normalized_depth=True,
)
print("✓ GeometryPipeline initialized")

# Initialize Matching Pipeline (includes the backbone model)
matchingPipeline = MatchingPipeline(
    config=config,
    model=None,  # Will create new model from scratch
    device=device,
)
print(f"✓ MatchingPipeline initialized")
print(f"  - Patch size: {matchingPipeline.patch_size}")
print(f"  - Embedding dim: {matchingPipeline.embed_dim}")
print(f"  - Sequence length: {matchingPipeline.seq_len}")

# Initialize Loss Function
from losses import TripletLoss

loss_fn = TripletLoss()
print("✓ Loss function initialized")

Using device: cuda
✓ GeometryPipeline initialized
✓ MatchingPipeline initialized
  - Patch size: 16
  - Embedding dim: 768
  - Sequence length: 576
✓ Loss function initialized


In [6]:
matchingPipeline.model.fromArtifact(
    pth_namestring="/home/arota/Match/checkpoints/weights_best.pt"
)

In [7]:
# Extract tensors from sample
framestack = sample["framestack"].to(device)  # [B, T, C, H, W]
camera_pose_gt = sample["Ts2t"].to(device)  # [B, 6]
batch_size = framestack.shape[0]

print(f"Framestack shape: {framestack.shape}")
print(f"Camera pose GT shape: {camera_pose_gt.shape}")
print(f"Batch size: {batch_size}")

# Handle intrinsics if available
if sample.get("intrinsics") is not None:
    K = sample["intrinsics"].to(device)
    print(f"Intrinsics shape: {K.shape}")
else:
    K = None
    print("No intrinsics in sample, will be inferred by geometry pipeline")

# Handle fundamental matrix if available
if sample.get("fundamental") is not None:
    fundamental_gt = sample["fundamental"].to(device)
    print(f"Fundamental GT shape: {fundamental_gt.shape}")
else:
    fundamental_gt = torch.zeros(batch_size, 3, 3, device=device)
    print("No fundamental matrix in sample, using dummy")

Framestack shape: torch.Size([1, 2, 3, 384, 384])
Camera pose GT shape: torch.Size([1, 6])
Batch size: 1
No intrinsics in sample, will be inferred by geometry pipeline
Fundamental GT shape: torch.Size([1, 1, 3, 3])


In [8]:
# Compute depth and intrinsics using MoGe-2
depthstack, _, K_moge = geometryPipeline.compute_geometry(framestack)

# Use inferred intrinsics if not provided in sample
if sample.get("intrinsics") is None:
    K = K_moge[:, 0]  # Use intrinsics from first frame

# Rescale depth to meaningful range
depthstack = depthstack * config.DEPTH_SCALE_FACTOR + config.DEPTH_BIAS_FACTOR

print(f"Depthstack shape: {depthstack.shape}")
print(f"Intrinsics K shape: {K.shape}")
print(f"Depth range: [{depthstack.min().item():.3f}, {depthstack.max().item():.3f}]")
print(f"Intrinsics sample (batch 0):\n{K[0]}")

Depthstack shape: torch.Size([1, 2, 1, 384, 384])
Intrinsics K shape: torch.Size([1, 3, 3])
Depth range: [30.000, 90.000]
Intrinsics sample (batch 0):
tensor([[377.9707,   0.0000, 192.0000],
        [  0.0000, 377.9707, 192.0000],
        [  0.0000,   0.0000,   1.0000]], device='cuda:0')


In [9]:
import augmentation as aug

# Apply geometric augmentation (disabled for debugging, set p=0.0)
# Set p to config value if you want to test with augmentation
framestack_aug, camera_pose_aug, depthstack_aug = aug.geometric_augmentation(
    framestack.clone(),
    camera_pose_gt.clone(),
    depthstack.clone(),
    p=0.0,  # Set to 0.0 for debugging, use config.AUGMENTATION_PROBABILITY["GEOMETRIC"] for training
)

print(f"Framestack shape after augmentation: {framestack_aug.shape}")
print(f"Camera pose shape after augmentation: {camera_pose_aug.shape}")
print(f"Depthstack shape after augmentation: {depthstack_aug.shape}")
print(
    f"Depth range after aug: [{depthstack_aug.min().item():.3f}, {depthstack_aug.max().item():.3f}]"
)

Framestack shape after augmentation: torch.Size([1, 2, 3, 384, 384])
Camera pose shape after augmentation: torch.Size([1, 6])
Depthstack shape after augmentation: torch.Size([1, 2, 1, 384, 384])
Depth range after aug: [30.000, 90.000]


## Step 5: Synthesize Ground Truth (Warping)

In [10]:
# Synthesize ground truth by warping source to target
gt_output = matchingPipeline.synthethize_ground_truth(
    framestack_aug,
    K,
    camera_pose_aug,
    depthstack_aug[:, 0],  # Use depth of first frame (source)
)

warped = gt_output["warped"]
source_matched_points = gt_output["source_matched_points"]
target_matched_points_true = gt_output["target_matched_points_true"]
embedding_mask = gt_output["embedding_mask"]

print(f"Warped image shape: {warped.shape}")
print(f"Source matched points shape: {source_matched_points.shape}")
print(f"GT shape: {target_matched_points_true.shape}")
print(f"Embedding mask shape: {embedding_mask.shape}")


Warped image shape: torch.Size([1, 3, 384, 384])
Source matched points shape: torch.Size([1, 32, 2])
GT shape: torch.Size([1, 32, 2])
Embedding mask shape: torch.Size([1, 768, 576])


In [11]:
# Create synthetic framestack with warped target image
synthetic_framestack = framestack_aug.clone()
synthetic_framestack[:, 1] = warped.clone()

print(f"Synthetic framestack shape: {synthetic_framestack.shape}")

# Apply color augmentation (disabled for debugging, set p=0.0)
synthetic_framestack_aug, camera_pose_final = aug.color_augmentation(
    synthetic_framestack.clone(),
    camera_pose_aug.clone(),
    p=0.0,  # Set to 0.0 for debugging, use config.AUGMENTATION_PROBABILITY["COLOR"] for training
    target_only=True,
)

print(f"Synthetic framestack after color aug: {synthetic_framestack_aug.shape}")
print(f"Camera pose (final) shape: {camera_pose_final.shape}")

Synthetic framestack shape: torch.Size([1, 2, 3, 384, 384])
Synthetic framestack after color aug: torch.Size([1, 2, 3, 384, 384])
Camera pose (final) shape: torch.Size([1, 6])


In [12]:
descriptors = matchingPipeline.model(synthetic_framestack_aug)

print(f"Source DINOv2-Features shape: {descriptors['source_embedding'].shape}")
print(f"Target DINOv2-Features shape: {descriptors['target_embedding'].shape}")
print(f"Source learned desctiptor shape: {descriptors['source_embedding_match'].shape}")
print(f"Target learned desctiptor shape: {descriptors['target_embedding_match'].shape}")

Source DINOv2-Features shape: torch.Size([1, 768, 576])
Target DINOv2-Features shape: torch.Size([1, 768, 576])
Source learned desctiptor shape: torch.Size([1, 768, 576])
Target learned desctiptor shape: torch.Size([1, 768, 576])


In [13]:
# Mine triplets (anchor, positive, negative) for training
triplets = matchingPipeline.mine_triplets(
    descriptors,
    source_matched_points,
    target_matched_points_true,
    embedding_mask,
)

A = triplets.get("anchor")  # Anchor embeddings
P = triplets.get("positive")  # Positive embeddings
N = triplets.get("negative")  # Negative embeddings

print(f"Anchor shape: {A.shape}")
print(f"Positive shape: {P.shape}")
print(f"Negative shape: {N.shape}")
print(f"Number of triplets mined: {len(A)}")
print(f"Triplets per batch element: {len(A) / batch_size:.1f}")

Anchor shape: torch.Size([30, 768])
Positive shape: torch.Size([30, 768])
Negative shape: torch.Size([30, 768])
Number of triplets mined: 30
Triplets per batch element: 30.0


## Step 9: Compute Loss

In [14]:
# Compute triplet loss
loss_tensor = loss_fn(A, P, N)
loss_value = loss_tensor.item()

print(f"Loss value: {loss_value:.6f}")

# Compute distances for analysis
with torch.no_grad():
    d_ap = torch.norm(A - P, p=2, dim=1).mean().item()  # Anchor-Positive distance
    d_an = torch.norm(A - N, p=2, dim=1).mean().item()  # Anchor-Negative distance

print(f"Average Anchor-Positive distance: {d_ap:.4f}")
print(f"Average Anchor-Negative distance: {d_an:.4f}")
print(f"Distance margin: {d_an - d_ap:.4f}")

Loss value: 0.813336
Average Anchor-Positive distance: 1187.8756
Average Anchor-Negative distance: 1428.2330
Distance margin: 240.3574


In [15]:
# Compute pixel correspondences between frames
correspondences = matchingPipeline.compute_correspondences(
    descriptors,
    synthetic_framestack_aug,
    embedding_mask,
)

source_pixels_matched = correspondences["source_pixels_matched"]
target_pixels_matched = correspondences["target_pixels_matched"]
batch_idx_match = correspondences["batch_idx_match"]
descriptor_scores = correspondences["descriptor_scores"]
refinement_scores = correspondences["refinement_scores"]
sim_matrix = correspondences["sim_matrix"]

print(f"Source pixels matched shape: {source_pixels_matched.shape}")
print(f"Target pixels matched shape: {target_pixels_matched.shape}")
print(f"Batch indices shape: {batch_idx_match.shape}")
print(f"Number of matches: {len(source_pixels_matched)}")
print(f"Descriptor scores shape: {descriptor_scores.shape}")
print(f"Refinement scores shape: {refinement_scores.shape}")
print(f"Similarity matrix shape: {sim_matrix.shape}")

Source pixels matched shape: torch.Size([500, 2])
Target pixels matched shape: torch.Size([500, 2])
Batch indices shape: torch.Size([500])
Number of matches: 500
Descriptor scores shape: torch.Size([500])
Refinement scores shape: torch.Size([500])
Similarity matrix shape: torch.Size([1, 576, 576])


In [16]:
# Estimate fundamental matrix using RANSAC
ransac_output = matchingPipeline.RANSAC(
    source_pixels_matched,
    target_pixels_matched,
    batch_idx_match,
)

fundamental_pred = ransac_output["F"]
inliers = ransac_output["inliers"]
epipolar_scores = ransac_output["scores"]

print(f"Fundamental matrix predicted shape: {fundamental_pred.shape}")
print(f"Inliers shape: {inliers.shape}")
print(f"Epipolar scores shape: {epipolar_scores.shape}")
print(f"Number of inliers: {inliers.sum().item()}/{len(inliers)}")
print(f"Inlier percentage: {100.0 * inliers.sum().item() / len(inliers):.2f}%")
print(f"Fundamental matrix (batch 0):\n{fundamental_pred[0]}")

Fundamental matrix predicted shape: torch.Size([1, 3, 3])
Inliers shape: torch.Size([500])
Epipolar scores shape: torch.Size([500])
Number of inliers: 194/500
Inlier percentage: 38.80%
Fundamental matrix (batch 0):
tensor([[-3.5444e-06, -1.2722e-04,  3.2196e-02],
        [ 4.6209e-05,  2.8289e-04,  3.6858e-01],
        [-2.1067e-02, -4.2364e-01,  8.2656e-01]], device='cuda:0')


In [17]:
# Combine descriptor, refinement, and epipolar scores
scores = matchingPipeline.combine_scores(
    descriptor_scores,
    refinement_scores,
    epipolar_scores,
    config.SCORE_WEIGHTS,
)

print(f"Combined scores shape: {scores.shape}")
print(f"Score range: [{scores.min().item():.4f}, {scores.max().item():.4f}]")
print(f"Mean score: {scores.mean().item():.4f}")
print(f"Median score: {scores.median().item():.4f}")

Combined scores shape: torch.Size([500])
Score range: [0.0757, 0.9161]
Mean score: 0.3507
Median score: 0.3091


In [18]:
# Retrieve pseudo-ground truth for the actual matched pixels
gt_output_eval = matchingPipeline.synthethize_ground_truth(
    synthetic_framestack_aug,
    K,
    camera_pose_final,
    depthstack_aug[:, 0],
    source_pixels_matched,
    batch_idx_match,
)

warped_eval = gt_output_eval["warped"]
source_pixels_matched_eval = gt_output_eval["source_matched_points"]
true_pixels_matched = gt_output_eval["target_matched_points_true"]
embedding_mask_eval = gt_output_eval["embedding_mask"]

print(f"True pixels matched shape: {true_pixels_matched.shape}")
print(
    f"Number of valid matches: {embedding_mask_eval.sum().int()}/{embedding_mask_eval.numel()} ({100 * embedding_mask_eval.sum().item() / embedding_mask_eval.numel():.2f}%)"
)

# Compute pixel error
pixel_error = torch.norm(target_pixels_matched - true_pixels_matched, p=2, dim=1)
print(f"Pixel correspondence error:")
print(f"  Mean: {pixel_error.mean().item():.2f} pixels")
print(f"  Median: {pixel_error.median().item():.2f} pixels")
print(f"  Max: {pixel_error.max().item():.2f} pixels")

True pixels matched shape: torch.Size([500, 2])
Number of valid matches: 291072/442368 (65.80%)
Pixel correspondence error:
  Mean: 33.72 pixels
  Median: 11.35 pixels
  Max: 317.72 pixels


In [19]:
# Compute all metrics
metrics = matchingPipeline.compute_metrics(
    matchingPipeline,
    source_pixels_matched,
    target_pixels_matched,
    true_pixels_matched,
    batch_idx_match,
    scores,
    fundamental_pred,
    fundamental_gt,
)

# Convert tensors to scalars for display
metrics = {
    k: (v.item() if isinstance(v, torch.Tensor) else v) for k, v in metrics.items()
}

print("=== METRICS SUMMARY ===")
for key, value in metrics.items():
    if value is not None:
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")

=== METRICS SUMMARY ===
Precision: 0.9742
Recall: 0.5286
AUCPR: 0.9750
EpipolarError: 35.6157
FundamentalError: 0.4710
MDistMean: 33.7227


In [ ]:
# Visualize matches for inspection
batch_idx_to_visualize = 0
batch_filter = batch_idx_match == batch_idx_to_visualize

# Visualize pixel matches
viewComparePixelMatches(
    img1=synthetic_framestack_aug[batch_idx_to_visualize, 0],
    img2=synthetic_framestack_aug[batch_idx_to_visualize, 1],
    pts1=source_pixels_matched[batch_filter],
    pts2=target_pixels_matched[batch_filter],
    pts2_true=true_pixels_matched[batch_filter],
    scores=scores[batch_filter],
    topk=50,
    use_actual_topk=True,
)

NameError: name 'viewComparePixelMatches' is not defined